# Phase 1 — Testing

Verifies the data pipeline end-to-end:
1. Ingest `headphones.json` into ChromaDB via the ingest script
2. Inspect the stored data
3. Test semantic queries

## 0 — Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
SRC = REPO_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repo root:", REPO_ROOT)
print("src exists:", SRC.exists())

Repo root: C:\Coding\AI-Engineering\ChatShop
src exists: True


## 1 — Ingest

Delete any existing ChromaDB and re-ingest from `data/headphones.json`.

In [2]:
import shutil

from chatshop.config import settings
from chatshop.data.cleaner import clean_headphones
from chatshop.data.loader import load_json
from chatshop.embeddings.embedder import Embedder
from chatshop.vectorstore.chroma import ChromaStore

chroma_path = REPO_ROOT / settings.chroma_persist_dir
if chroma_path.exists():
    shutil.rmtree(chroma_path)
    print(f"Deleted existing ChromaDB at {chroma_path}")

raw = load_json(REPO_ROOT / "data" / "headphones.json")
products = clean_headphones(raw)
print(f"{len(raw)} raw → {len(products)} clean products")

embedder = Embedder()
vectors = embedder.encode([p.to_document_text() for p in products])

store = ChromaStore()
store.upsert(products, vectors)
print(f"Ingested {store.count()} products into ChromaDB")

100 raw → 100 clean products
Ingested 100 products into ChromaDB


## 2 — Inspect ChromaDB

In [8]:
import json

from chatshop.vectorstore.chroma import ChromaStore

store = ChromaStore()
print(f"Documents in collection: {store.count()}")

raw = store._collection.peek(limit=1)
print("\nMetadata for first document:")
print(json.dumps(raw["metadatas"][0], indent=2))

Documents in collection: 100

Metadata for first document:
{
  "battery_hours": 30,
  "description": "Experience unparalleled noise cancellation with the Sony WH-1000XM5. Featuring eight microphones and two processors, these flagship over-ear headphones create an immersive soundscape that blocks out the world. The 30mm drivers deliver rich, detailed audio with deep bass and crystalline highs. An ergonomic headband with ultra-plush ear cushions ensures all-day comfort on long flights or marathon work sessions. Industry-leading 30-hour battery life keeps the music playing, while Speak-to-Chat tech",
  "waterproof_rating": "",
  "brand": "Sony",
  "use_cases": "travel, office, studio",
  "title": "Sony WH-1000XM5",
  "driver_size_mm": 30.0,
  "wireless": true,
  "anc": true,
  "price": 349.0,
  "type": "over-ear"
}


In [9]:
from chatshop.embeddings.embedder import Embedder

embedder = Embedder()


def show_results(products, label=""):
    if label:
        print(f"\n{'─'*55}\n {label}\n{'─'*55}")
    for i, p in enumerate(products, 1):
        price = f"${p.price:.0f}" if p.price else "N/A"
        flags = []
        if p.wireless:
            flags.append("wireless")
        if p.anc:
            flags.append("ANC")
        if p.use_cases:
            flags.append(", ".join(p.use_cases))
        print(f"[{i}] {p.title}")
        print(f"     {price} | {p.type} | {' | '.join(flags)}")

In [10]:
# Semantic + metadata filter: ANC, wireless, under $200
where = {
    "$and": [
        {"wireless": {"$eq": True}},
        {"anc": {"$eq": True}},
        {"price": {"$lte": 200}},
        {"price": {"$gt": 0}},
    ]
}
results = store.query(embedder.encode_one("commute noise cancelling"), top_k=5, where=where)
show_results(results, "ANC + wireless + price ≤ $200")


───────────────────────────────────────────────────────
 ANC + wireless + price ≤ $200
───────────────────────────────────────────────────────
[1] Audio-Technica Nova 3
     $99 | over-ear | wireless | ANC | travel, office
[2] Sony WH-CH720N
     $149 | over-ear | wireless | ANC | travel, office
[3] Sennheiser HD 450BT
     $149 | over-ear | wireless | ANC | travel, office
[4] Anker Soundcore Liberty 4 NC
     $79 | in-ear | wireless | ANC | travel, office, sport
[5] JBL Tune 760NC
     $99 | over-ear | wireless | ANC | travel, office


In [11]:
# Sport / gym — in-ear, wireless
where_sport = {
    "$and": [
        {"wireless": {"$eq": True}},
        {"type": {"$eq": "in-ear"}},
    ]
}
results = store.query(embedder.encode_one("earbuds for running and gym"), top_k=5, where=where_sport)
show_results(results, "in-ear + wireless (sport)")


───────────────────────────────────────────────────────
 in-ear + wireless (sport)
───────────────────────────────────────────────────────
[1] Sennheiser Momentum Sport
     $279 | in-ear | wireless | ANC | sport
[2] Bose SoundSport Free
     $129 | in-ear | wireless | sport
[3] Jabra Endurance Peak 3
     $79 | in-ear | wireless | sport
[4] JLab JLab Epic Air Sport ANC Gen 2
     $79 | in-ear | wireless | ANC | sport, travel
[5] Jabra Elite Active 75t
     $149 | in-ear | wireless | ANC | sport, travel
